# Part 1 - Extend the Engine

### tensor.py

In [1]:
#---------tensor.py---------#
#--------Andrew_martin------#



import numpy as np


class Tensor:
    def __init__(self, data, autograd=False, creators=None,
                creation_op=None, id=None):
        self.data = np.array(data)
        self.autograd = autograd
        self.grad = None
        self.id = id if id is not None else np.random.randint(0, 100000)
        self.creators = creators
        self.creation_op = creation_op
        self.children = {}
        if creators is not None:
            for c in creators:
                c.children[self.id] = c.children.get(self.id, 0) + 1

    def all_children_accounted_for(self):
        return all(cnt == 0 for cnt in self.children.values())

    def backward(self, grad=None, grad_origin=None):
        if not self.autograd:
            return
        if grad is None:
            grad = Tensor(np.ones_like(self.data))
        if grad_origin is not None:
            if self.children[grad_origin.id] == 0:
                raise Exception("cannot backprop more than once")
            self.children[grad_origin.id] -= 1

        if self.grad is None:
            self.grad = grad
        else:
            self.grad += grad
        assert grad.autograd == False # grads carry no grads

        if self.creators is not None and \
            (self.all_children_accounted_for() or grad_origin is None):
            if self.creation_op == "add":
                self.creators[0].backward(Tensor(self.grad.data), self)
                self.creators[1].backward(Tensor(self.grad.data), self)
            if self.creation_op == "sub":
                self.creators[0].backward(Tensor(self.grad.data), self)
                self.creators[1].backward(
                Tensor(self.grad.__neg__().data), self)
            if self.creation_op == "mul":
                self.creators[0].backward(self.grad * self.creators[1], self)
                self.creators[1].backward(self.grad * self.creators[0], self)
            if self.creation_op == "mm":
                c0, c1 = self.creators
                c0.backward(Tensor(self.grad.data.dot(c1.data.T)), self)
                c1.backward(Tensor(c0.data.T.dot(self.grad.data)), self)
            if self.creation_op == "transpose":
                self.creators[0].backward(self.grad.transpose(), self)
            if "sum" in self.creation_op:
                dim = int(self.creation_op.split("_")[1])
                ds = self.creators[0].data.shape[dim]
                self.creators[0].backward(self.grad.expand(dim, ds), self)
            if "expand" in self.creation_op:
                dim = int(self.creation_op.split("_")[1])
                self.creators[0].backward(self.grad.sum(dim), self)
            if self.creation_op == "neg":
                self.creators[0].backward(self.grad.__neg__(), self)
            if self.creation_op == "sigmoid":
                out = self.data # the activation's own output
                self.creators[0].backward(
                    Tensor(self.grad.data * (out * (1 - out))), self)
            if self.creation_op == "tanh":
                out = self.data
                self.creators[0].backward(
                    Tensor(self.grad.data * (1 - out * out)), self)
            if self.creation_op == "index_select":
                new_grad = np.zeros_like(self.creators[0].data)
                indices_ = self.index_select_indices.data.flatten()
                grad_ = self.grad.data.reshape(len(indices_), -1)
                for i in range(len(indices_)):
                    new_grad[indices_[i]] += grad_[i]
                self.creators[0].backward(Tensor(new_grad), self)
            if self.creation_op == "cross_entropy":
                dx = (self.softmax_output - self.target_dist) / len(self.target_dist)
                self.creators[0].backward(Tensor(dx), self)

    def __add__(self, other):
        if self.autograd and other.autograd:
            return Tensor(self.data + other.data, autograd=True,
                          creators=[self, other], creation_op="add")
        return Tensor(self.data + other.data)

    def __neg__(self):
        if self.autograd:
            return Tensor(self.data * -1, autograd=True,
                        creators=[self], creation_op="neg")
        return Tensor(self.data * -1)

    def __sub__(self, other):
        if self.autograd and other.autograd:
            return Tensor(self.data - other.data, autograd=True,
                    creators=[self, other], creation_op="sub")
        return Tensor(self.data - other.data)

    def __mul__(self, other):
        if self.autograd and other.autograd:
            return Tensor(self.data * other.data, autograd=True,
                        creators=[self, other], creation_op="mul")
        return Tensor(self.data * other.data)

    def sum(self, dim):
        if self.autograd:
            return Tensor(self.data.sum(dim), autograd=True,
                        creators=[self], creation_op="sum_" + str(dim))
        return Tensor(self.data.sum(dim))

    def expand(self, dim, copies):
        trans_cmd = list(range(0, len(self.data.shape)))
        trans_cmd.insert(dim, len(self.data.shape))
        new_data = self.data.repeat(copies).reshape(
        list(self.data.shape) + [copies]).transpose(trans_cmd)
        if self.autograd:
            return Tensor(new_data, autograd=True,
                creators=[self], creation_op="expand_" + str(dim))
        return Tensor(new_data)

    def transpose(self):
        if self.autograd:
            return Tensor(self.data.transpose(), autograd=True,
                    creators=[self], creation_op="transpose")
        return Tensor(self.data.transpose())

    def mm(self, x):
        if self.autograd:
            return Tensor(self.data.dot(x.data), autograd=True,
                creators=[self, x], creation_op="mm")
        return Tensor(self.data.dot(x.data))

    def sigmoid(self):
        if self.autograd:
            return Tensor(1 / (1 + np.exp(-self.data)), autograd=True,
                creators=[self], creation_op="sigmoid")
        return Tensor(1 / (1 + np.exp(-self.data)))

    def tanh(self):
        if self.autograd:
            return Tensor(np.tanh(self.data), autograd=True,
                creators=[self], creation_op="tanh")
        return Tensor(np.tanh(self.data))
   
    def index_select(self, indices):
        if self.autograd:
            out = Tensor(self.data[indices.data], autograd=True,
                creators=[self], creation_op="index_select")
            out.index_select_indices = indices
            return out
        return Tensor(self.data[indices.data])

    def softmax(self):
        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp, axis=len(self.data.shape) - 1,
                                       keepdims=True)
        return softmax_output

    def cross_entropy(self, target_indices):
        temp = np.exp(self.data)
        softmax_output = temp / np.sum(temp, axis=len(self.data.shape) - 1,
                                       keepdims=True)
        t = target_indices.data.flatten()
        p = softmax_output.reshape(len(t), -1)
        target_dist = np.eye(p.shape[1])[t]
        loss = -(np.log(p) * target_dist).sum(1).mean()

        if self.autograd:
            out = Tensor(loss, autograd=True, creators=[self],
                         creation_op="cross_entropy")
            out.softmax_output = softmax_output
            out.target_dist = target_dist
            return out
        return Tensor(loss)

    def __repr__(self):
        return str(self.data)


if __name__ == "__main__":
    # Example 1: b is used twice, so its two contributions must add up.
    a = Tensor([1, 2, 3, 4, 5], autograd=True)
    b = Tensor([2, 2, 2, 2, 2], autograd=True)
    c = Tensor([5, 4, 3, 2, 1], autograd=True)
    d = a + b
    e = b + c
    f = d + e
    f.backward(Tensor([1, 1, 1, 1, 1]))
    print("b gradient (expected all 2s):", b.grad.data)

    # Example 2: s must collect both contributions before passing them on.
    x = Tensor([1, 2, 3, 4, 5], autograd=True)
    y = Tensor([1, 1, 1, 1, 1], autograd=True)
    s = x + y
    loss = (s + a) + (s + c)
    loss.backward(Tensor([1, 1, 1, 1, 1]))
    print("x gradient (expected all 2s):", x.grad.data)

b gradient (expected all 2s): [2 2 2 2 2]
x gradient (expected all 2s): [2 2 2 2 2]


# Part 2 - The Lookup and the Loss

### layers.py

In [2]:
#---------layers.py----------#
#-------Andrew_Martin--------#
import numpy as np

from tensor import Tensor


class Layer:
    # Base class: every layer keeps a list of the Tensors it learns.

    def __init__(self):
        self.parameters = []

    def get_parameters(self):
        return self.parameters


class Linear(Layer):
    # A fully connected layer: output = input @ weight (+ bias).

    def __init__(self, n_in, n_out, bias=True):
        super().__init__()
        # Scaled random start so the signal neither dies nor explodes.
        W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.weight = Tensor(W, autograd=True)
        self.parameters.append(self.weight)

        # With bias=False, no bias Tensor is created at all (creating one
        # would also draw a Tensor id and shift the seeded random stream).
        self.use_bias = bias
        if bias:
            self.bias = Tensor(np.zeros(n_out), autograd=True)
            self.parameters.append(self.bias)

    def forward(self, input):
        if self.use_bias:
            # bias is one row; expand copies it once per row of the batch.
            return input.mm(self.weight) + self.bias.expand(0, len(input.data))
        return input.mm(self.weight)


class Sequential(Layer):
    # A stack of layers run one after another.

    def __init__(self, layers=None):
        super().__init__()
        self.layers = layers if layers is not None else []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def get_parameters(self):
        params = []
        for layer in self.layers:
            params += layer.get_parameters()
        return params


class Tanh(Layer):
    """Squash each value into (-1, 1). No parameters."""

    def forward(self, input):
        return input.tanh()


class Sigmoid(Layer):
    """Squash each value into (0, 1). No parameters."""

    def forward(self, input):
        return input.sigmoid()


class MSELoss(Layer):
    """Squared error summed over the batch (summed, not averaged)."""

    def forward(self, pred, target):
        return ((pred - target) * (pred - target)).sum(0)


class Embedding(Layer):
    # A learnable lookup table: a weight you index instead of multiply.

    def __init__(self, vocab_size, dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim

        W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.weight = Tensor(W, autograd=True)
        self.parameters.append(self.weight)

    def forward(self, input):
        # No gradient code: index_select's backward carries it.
        return self.weight.index_select(input)


class CrossEntropyLoss(object):
    # Thin wrapper; owns no parameters, so it is not a Layer.

    def forward(self, input, target):
        return input.cross_entropy(target)


class LSTMCell(Layer):
    # Three gates, a candidate, and a cell-state belt.

    def __init__(self, n_inputs, n_hidden, n_output):
        super().__init__()
        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        self.xf = Linear(n_inputs, n_hidden)                # input -> gate (with bias)
        self.xi = Linear(n_inputs, n_hidden)
        self.xo = Linear(n_inputs, n_hidden)
        self.xc = Linear(n_inputs, n_hidden)
        self.hf = Linear(n_hidden, n_hidden, bias=False)    # hidden -> gate (no bias)
        self.hi = Linear(n_hidden, n_hidden, bias=False)
        self.ho = Linear(n_hidden, n_hidden, bias=False)
        self.hc = Linear(n_hidden, n_hidden, bias=False)
        self.w_ho = Linear(n_hidden, n_output, bias=False)  # cell view -> logits

        # Register every sub-layer's parameters, in the order built above.
        for layer in [self.xf, self.xi, self.xo, self.xc,
                      self.hf, self.hi, self.ho, self.hc, self.w_ho]:
            self.parameters += layer.get_parameters()

    def forward(self, input, hidden):
        prev_hidden, prev_cell = hidden[0], hidden[1]
        f = (self.xf.forward(input) + self.hf.forward(prev_hidden)).sigmoid()
        i = (self.xi.forward(input) + self.hi.forward(prev_hidden)).sigmoid()
        o = (self.xo.forward(input) + self.ho.forward(prev_hidden)).sigmoid()
        g = (self.xc.forward(input) + self.hc.forward(prev_hidden)).tanh()
        c = (f * prev_cell) + (i * g)   # keep f of the old, write i of the new
        h = o * c.tanh()                # expose a gated view of the memory
        return self.w_ho.forward(h), (h, c)

    def init_hidden(self, batch_size=1):
        # Zeros, with feature 0 nudged to 1 to break symmetry (as Trask does).
        init_hidden = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        init_cell = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        init_hidden.data[:, 0] += 1
        init_cell.data[:, 0] += 1
        return (init_hidden, init_cell)


# Part 3 - Vanishing, Exploding, and the Gate That Holds

### exploding_gradients.py

In [3]:

# Carter Owens


import numpy as np

W = np.array([[1,4], [4, 1]])

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))



def relu(x):
    return np.maximum(0.0, x)


def vanishing():
    activation = sigmoid(np.array([1, 0.01]))
    acts = []

    for _ in range(10):
        activation = sigmoid(activation.dot(W))
        acts.append(activation)
   
    gradient = np.ones(2)
    gradients = []

    for act in reversed(acts):
        gradient = (gradient * act * (1 - act)).dot(W.T)
        gradients.append(gradient)

    return gradients


def exploding():
    activation = sigmoid(np.array([1, 0.01]))
    acts = []

    for _ in range(10):
        activation = relu(activation.dot(W))
        acts.append(activation)
   
    gradient = np.ones(2)
    gradients = []

    for act in reversed(acts):
        gradient = (gradient *(act > 0)).dot(W.T)
        gradients.append(gradient)
    
    return gradients




def lstm_cell_state_horizon(steps=50, forget_gate=True, seed=0):
    rng = np.random.RandomState(seed)
    c, norms = np.zeros(16), []
    for _ in range(steps):
        x = rng.randn(16)
        f_pre = rng.randn(16) # always drawn: same writes either way
        f = sigmoid(f_pre) if forget_gate else np.ones(16)
        i = sigmoid(rng.randn(16))
        c = f * c + i * np.tanh(x) # keep f of the old, write i of the new
        norms.append(float(np.max(np.abs(c))))
    return norms


if __name__ == "__main__":
    print("Vanishing:", [float(np.max(np.abs(g))) for g in vanishing()])
    print("Exploding:", [float(np.max(np.abs(g))) for g in exploding()])
    print("LSTM with forget gate:", lstm_cell_state_horizon(forget_gate=True)[-1])
    print("LSTM without forget gate:", lstm_cell_state_horizon(forget_gate=False)[-1])


Vanishing: [0.03439551741586478, 0.0011830516183190963, 4.069167255630264e-05, 1.399611146713329e-06, 4.8140364304527505e-08, 1.6558276535062037e-09, 5.696826748386598e-11, 1.9751791957978857e-12, 8.453875971037248e-14, 2.16938982636746e-14]
Exploding: [5.0, 25.0, 125.0, 625.0, 3125.0, 15625.0, 78125.0, 390625.0, 1953125.0, 9765625.0]
LSTM with forget gate: 0.9607371541355537
LSTM without forget gate: 5.178734457951545


# Part 4 - The Hero; LSTMCell

### layers.py

In [4]:
#---------layers.py----------#
#-------Andrew_Martin--------#
import numpy as np

from tensor import Tensor


class Layer:
    # Base class: every layer keeps a list of the Tensors it learns.

    def __init__(self):
        self.parameters = []

    def get_parameters(self):
        return self.parameters


class Linear(Layer):
    # A fully connected layer: output = input @ weight (+ bias).

    def __init__(self, n_in, n_out, bias=True):
        super().__init__()
        # Scaled random start so the signal neither dies nor explodes.
        W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
        self.weight = Tensor(W, autograd=True)
        self.parameters.append(self.weight)

        # With bias=False, no bias Tensor is created at all (creating one
        # would also draw a Tensor id and shift the seeded random stream).
        self.use_bias = bias
        if bias:
            self.bias = Tensor(np.zeros(n_out), autograd=True)
            self.parameters.append(self.bias)

    def forward(self, input):
        if self.use_bias:
            # bias is one row; expand copies it once per row of the batch.
            return input.mm(self.weight) + self.bias.expand(0, len(input.data))
        return input.mm(self.weight)


class Sequential(Layer):
    # A stack of layers run one after another.

    def __init__(self, layers=None):
        super().__init__()
        self.layers = layers if layers is not None else []

    def add(self, layer):
        self.layers.append(layer)

    def forward(self, input):
        for layer in self.layers:
            input = layer.forward(input)
        return input

    def get_parameters(self):
        params = []
        for layer in self.layers:
            params += layer.get_parameters()
        return params


class Tanh(Layer):
    """Squash each value into (-1, 1). No parameters."""

    def forward(self, input):
        return input.tanh()


class Sigmoid(Layer):
    """Squash each value into (0, 1). No parameters."""

    def forward(self, input):
        return input.sigmoid()


class MSELoss(Layer):
    """Squared error summed over the batch (summed, not averaged)."""

    def forward(self, pred, target):
        return ((pred - target) * (pred - target)).sum(0)


class Embedding(Layer):
    # A learnable lookup table: a weight you index instead of multiply.

    def __init__(self, vocab_size, dim):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim

        W = (np.random.rand(vocab_size, dim) - 0.5) / dim
        self.weight = Tensor(W, autograd=True)
        self.parameters.append(self.weight)

    def forward(self, input):
        # No gradient code: index_select's backward carries it.
        return self.weight.index_select(input)


class CrossEntropyLoss(object):
    # Thin wrapper; owns no parameters, so it is not a Layer.

    def forward(self, input, target):
        return input.cross_entropy(target)


class LSTMCell(Layer):
    # Three gates, a candidate, and a cell-state belt.

    def __init__(self, n_inputs, n_hidden, n_output):
        super().__init__()
        self.n_inputs = n_inputs
        self.n_hidden = n_hidden
        self.n_output = n_output

        self.xf = Linear(n_inputs, n_hidden)                # input -> gate (with bias)
        self.xi = Linear(n_inputs, n_hidden)
        self.xo = Linear(n_inputs, n_hidden)
        self.xc = Linear(n_inputs, n_hidden)
        self.hf = Linear(n_hidden, n_hidden, bias=False)    # hidden -> gate (no bias)
        self.hi = Linear(n_hidden, n_hidden, bias=False)
        self.ho = Linear(n_hidden, n_hidden, bias=False)
        self.hc = Linear(n_hidden, n_hidden, bias=False)
        self.w_ho = Linear(n_hidden, n_output, bias=False)  # cell view -> logits

        # Register every sub-layer's parameters, in the order built above.
        for layer in [self.xf, self.xi, self.xo, self.xc,
                      self.hf, self.hi, self.ho, self.hc, self.w_ho]:
            self.parameters += layer.get_parameters()

    def forward(self, input, hidden):
        prev_hidden, prev_cell = hidden[0], hidden[1]
        f = (self.xf.forward(input) + self.hf.forward(prev_hidden)).sigmoid()
        i = (self.xi.forward(input) + self.hi.forward(prev_hidden)).sigmoid()
        o = (self.xo.forward(input) + self.ho.forward(prev_hidden)).sigmoid()
        g = (self.xc.forward(input) + self.hc.forward(prev_hidden)).tanh()
        c = (f * prev_cell) + (i * g)   # keep f of the old, write i of the new
        h = o * c.tanh()                # expose a gated view of the memory
        return self.w_ho.forward(h), (h, c)

    def init_hidden(self, batch_size=1):
        # Zeros, with feature 0 nudged to 1 to break symmetry (as Trask does).
        init_hidden = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        init_cell = Tensor(np.zeros((batch_size, self.n_hidden)), autograd=True)
        init_hidden.data[:, 0] += 1
        init_cell.data[:, 0] += 1
        return (init_hidden, init_cell)


# Part 5 - Train on the Old Tongue

### train_lstm.py

In [5]:
import numpy as np
import matplotlib.pyplot as plt

from tensor import Tensor
from layers import Embedding, LSTMCell, CrossEntropyLoss
from optim import SGD


np.random.seed(0)
embedding_size = 512
hidden_size = 512
batch_size = 16
bptt = 25
alpha = 0.032

text = open("data/qa_oldtongue.txt", encoding="utf-8").read()
vocab = sorted(set(text))
char_to_idx = {ch: i for i, ch in enumerate(vocab)}
data = np.array([char_to_idx[ch] for ch in text])

# Split the text into 16 continuous sequences
data = data[:len(data) // batch_size * batch_size]
data = data.reshape(batch_size, -1)

embedding = Embedding(len(vocab), embedding_size)
model = LSTMCell(embedding_size, hidden_size, len(vocab))
model.w_ho.weight.data *= 0
loss_fn = CrossEntropyLoss()

optimizer = SGD(
    embedding.get_parameters() + model.get_parameters(),
    alpha=alpha
)

hidden = model.init_hidden(batch_size)
total_loss = 0
total_steps = 0
perplexities = []

batch_num = 0
for start in range(0, data.shape[1] - 1, bptt):
    if batch_num == 60:
        break

    end = min(start + bptt, data.shape[1] - 1)

    hidden = (
        Tensor(hidden[0].data, autograd=True),
        Tensor(hidden[1].data, autograd=True)
    )

    chunk_loss = None

    for t in range(start, end):
        x = Tensor(data[:, t])
        target = Tensor(data[:, t + 1])

        x = embedding.forward(x)
        output, hidden = model.forward(x, hidden)
        loss = loss_fn.forward(output, target)

        chunk_loss = loss if chunk_loss is None else chunk_loss + loss

    chunk_loss.backward()
    optimizer.step()

    total_loss += float(chunk_loss.data)
    total_steps += end - start
    perplexity = np.exp(total_loss / total_steps)
    perplexities.append(perplexity)

    batch_num += 1
    print(f"Batch {batch_num}: perplexity = {perplexity:.2f}")

# Save the training curve for the final notebook
plt.plot(range(1, len(perplexities) + 1), perplexities)
plt.xlabel("Batch")
plt.ylabel("Running perplexity")
plt.title("LSTM Training")
plt.savefig("training_curve.png")
plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'data/qa_oldtongue.txt'